In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import getpass
import gc
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random
from tqdm.auto import tqdm
from timeit import default_timer as timer

device = "cuda" if torch.cuda.is_available() else "cpu"
a = str(device).title()
print(f"Using Device = {a}")

def print_state_end_time(start, end, device=device):
    return end - start


script_dir = os.path.dirname(os.path.abspath(os.getcwd()))

df_path = os.path.join(script_dir,"Network Datasets","Student Performance Score Dataset.csv")
df = pd.read_csv(df_path)

df = df.dropna()

x = df.drop(columns="final_score").values
y = df["final_score"].values.reshape(-1,1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y)

x_train, x_test, y_train ,y_test = train_test_split(x,y, test_size=0.2, random_state=23)
num_features = x_train.shape[1]

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

print(f"X = {x.shape}")
print(f"Y = {y.shape}")
print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")

user_account = getpass.getuser()
predictions_bar_chart = os.path.join(script_dir,"Graphs",f"Student Performance Score Regressor Network Predictions Bar Chart.png")

class Regressor(nn.Module):
    def __init__(self, input_dim =num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=1)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

def run_grid_search(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = nn.L1Loss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10}")
    print(f"="*65)
    losses = []

    start = timer()
    for lr in rates_list:
        for epoch in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = Regressor().to(device)   
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)     

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_loss = loss_fn(model(x_test), y_test)
                losses.append(loss)

            print(f"{lr:<15} | {epoch:<8} | {loss:<10.4f}")

            if test_loss < best_loss:
                best_loss = test_loss
                import copy
                best_state = copy.deepcopy(model.state_dict()) 

            del model, optimizer
            gc.collect()       
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")  
    print(f"● Worst Loss = {max(losses):<10.4f}")          
    end = timer()
    print_state_end_time(start, end)
    print(f"● Training Time = {print_state_end_time}\n")

    return best_state, best_loss,losses

epochs_list = [380,900,710, 450]
rates = [0.001, 0.0001, 0.2, 0.1]

for file in [predictions_bar_chart, f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Details.txt"]:
    if os.path.exists(file):
        try:os.remove(file)
        except OSError:pass

seed = float(input("-- Enter a Value for a Seed : ").strip())
best_state, best_loss, losses = run_grid_search(rates, epochs_list, seed)
torch.save(best_state, f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Parameters.pth")
fresh_model = Regressor().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

ac = {
    "Yes":1,
    "No":0
}

session_inputs = []
session_predictions = []

while True:
    raw_name_input = input("-- Enter your Name ( or type 'exit' to stop ): ").strip()
    
    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break
        
    user_name = raw_name_input.title()
    
    hours_studied = float(input("-- How many Hours did you Study for : ").strip())
    attendance_rate = float(input("-- What is your Attendance Rate : ").strip())
    previous_score = float(input("-- What was your Previous Score : ").strip())
    sleep_hours = float(input("-- How many Hours did you Sleep for : ").strip())
    extracurricular_activities = input("-- Do you do any Extra Curricular Activities (Yes/No) : ").strip().title()
    internet_quality = float(input("-- Enter your Internet Quality : ").strip())

    ex = ac.get(extracurricular_activities, 0)

    raw_array = np.array([[hours_studied, attendance_rate, previous_score, sleep_hours, ex, internet_quality]])
    scaled_array = x_scaler.transform(raw_array)
    scaled_tensor = torch.tensor(scaled_array, dtype=torch.float32).to(device)

    with torch.inference_mode():
        preds = fresh_model(scaled_tensor).cpu().numpy()

    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted Performance Score for you, {user_name} = {untransformed:.2f}")   

    session_inputs.append(f"{user_name}\n({round(hours_studied)} Hours Studied )")
    session_predictions.append(untransformed)

try:
    if session_inputs and session_predictions:
        plt.figure(figsize=(14,6))
        colors = ["red","blue","purple","orange","green"]
        bar_colors = [random.choice(colors) for _ in range(len(session_predictions))]
        plt.bar(session_inputs, session_predictions, color=bar_colors, edgecolor="black")
        plt.title("Predictions Bar Chart = Student Performance Scores")
        plt.xlabel("Students | Study Hours")
        plt.ylabel("Predicted Student Performance Scores")
        plt.grid()
        plt.legend(prop={"size":12,"weight":"bold"})
        plt.savefig(predictions_bar_chart)
        plt.close()
        print(f"Saved the Graph = {predictions_bar_chart}")
    else:
        print(f"Failed to Save Graph = No Inputs & Predictions")    
except BaseException as e:    
        print(f"Failed to Save Graph = {e}")

onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
if onedrive == "Yes":
    desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Student Performance Score Regressor Network Details.txt"
elif onedrive == "No":
    desktop_path = f"C:/Users/{user_account}/Desktop/Student Performance Score Regressor Network Details.txt"

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Student Performance Score Regressor Network Weights & Biases ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {best_loss:.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessefully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")    